# Hybrid Phishing Detection Workflow

This notebook combines the **fine-tuned Random Forest** (URL classifier) and the **fine-tuned BERT** (text classifier) into a single `UnifiedPhishingDetector`, and runs it against 12 example messages.

## How it works

1. **Extract URLs** from the message text using a regex.
2. **Text analysis** - strip the URLs, then score the remaining text with BERT.
3. **URL analysis** - build lexical/structural features for each URL and score them with the Random Forest.
4. **Three-way decision** using calibrated lower/upper thresholds:
   - `PHISHING` -> high confidence, block automatically.
   - `LEGITIMATE` -> high confidence, allow automatically.
   - `HUMAN_REVIEW` -> uncertain, escalate to an analyst.

## Decision methods

| Method | When it is used |
| --- | --- |
| `URL_ONLY` | Message contains one or more URLs and no other text |
| `TEXT_ONLY` | Message contains text but no URLs |
| `HYBRID` | Both text and URLs - a URL verdict can override the text verdict |

The hybrid overrides encode two design rules: a **phishing URL overrides safe text**, and a **trusted/safe URL overrides suspicious text**.

## Requirements

The detector loads two artefacts that are **not** included in this repository:

- `rf_test_finetuned.joblib` - the trained Random Forest model
- `bert_test/` - the fine-tuned BERT model

Update the paths in `UnifiedPhishingDetector.__init__` to point at your local copies before running.


In [ ]:
import pandas as pd
import re
import torch
from urllib.parse import urlparse
from joblib import load
from transformers import BertForSequenceClassification, BertTokenizer
from rf_finetuned_feature_extraction import extract_features_vectorized

class UnifiedPhishingDetector:
    def __init__(self):
        # Load Random Forest model
        self.url_model = load('/Users/abdulsalamali/School/Dissertation/final/rf_test_finetuned.joblib')
        
        # Load BERT model and tokenizer
        self.bert_model = BertForSequenceClassification.from_pretrained('/Users/abdulsalamali/School/Dissertation/final/bert_test')
        self.bert_tokenizer = BertTokenizer.from_pretrained('/Users/abdulsalamali/School/Dissertation/final/bert_test')
        
        # Load trusted domains
        trusted_df = pd.read_csv('/Users/abdulsalamali/School/Dissertation/datasets/trusted_domains.csv')
        self.trusted_domains = set(trusted_df['url'].str.lower())
        
        # Hardcoded thresholds
        self.thresholds = {
            'url': {'lower': 0.01, 'upper': 0.6355},
            'text': {'lower': 0.4756, 'upper': 0.8187}
        }
    
    def _extract_urls_from_text(self, text):
        url_pattern = r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\\(\\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+'
        return re.findall(url_pattern, text)
    
    def _predict_url_phishing(self, url):
        url_series = pd.Series([url])
        features_df = extract_features_vectorized(url_series, self.trusted_domains)
        
        # Check trusted domain override
        is_trusted = features_df['is_trusted_domain'].iloc[0] == 1
        if is_trusted:
            return 0.01
        
        prob = self.url_model.predict_proba(features_df)[0, 1]
        return prob
    
    def _predict_text_phishing(self, text):
        # Remove URLs from text before BERT analysis
        url_pattern = r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\\(\\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+'
        text_without_urls = re.sub(url_pattern, '', text)
        text_without_urls = ' '.join(text_without_urls.split())
        
        if not text_without_urls.strip():
            return 0.5
        
        inputs = self.bert_tokenizer(
            text_without_urls,
            truncation=True,
            padding="max_length",
            max_length=256,
            return_tensors="pt"
        )
        
        with torch.no_grad():
            outputs = self.bert_model(**inputs)
            predictions = torch.softmax(outputs.logits, dim=1)
            phishing_prob = predictions[0][1].item()
        
        return phishing_prob
    
    def predict_single(self, text):
        urls = self._extract_urls_from_text(text)
        
        result = {
            'text': text,
            'contains_urls': len(urls) > 0,
            'urls_found': urls,
            'url_analysis': [],
            'text_analysis': None,
            'final_prediction': None,
            'final_probability': None,
            'confidence_level': None,
            'analysis_method': "Hybrid"
        }

        # Text analysis
        url_pattern = r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\\(\\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+'
        text_without_urls = re.sub(url_pattern, '', text).strip()
        text_without_urls = ' '.join(text_without_urls.split())
        is_url_only = not text_without_urls
        
        text_prob, text_classification, text_confidence = 0.0, 'LEGITIMATE', 'LOW'
        
        if not is_url_only:
            text_prob = self._predict_text_phishing(text)
            text_threshold = self.thresholds['text']
            if text_prob >= text_threshold['upper']:
                text_classification, text_confidence = 'PHISHING', 'HIGH'
            elif text_prob > text_threshold['lower']:
                text_classification, text_confidence = 'HUMAN_REVIEW', 'UNCERTAIN'
            else:
                text_classification, text_confidence = 'LEGITIMATE', 'HIGH'
            
            result['text_analysis'] = {
                'probability': text_prob,
                'classification': text_classification,
                'confidence': text_confidence
            }
        else:
            result['text_analysis'] = {
                'note': 'N/A - No text content to analyse (URL-only input)'
            }

        # URL analysis
        max_url_prob, url_classification, url_confidence = 0.0, 'LEGITIMATE', 'LOW'
        if urls:
            url_probs = []
            url_threshold = self.thresholds['url']
            
            for url in urls:
                prob = self._predict_url_phishing(url)
                url_probs.append(prob)
                
                if prob <= url_threshold['lower']:
                    classification = 'LEGITIMATE'
                    confidence = 'HIGH'
                elif prob >= url_threshold['upper']:
                    classification = 'PHISHING'
                    confidence = 'HIGH'
                else:
                    classification = 'HUMAN_REVIEW'
                    confidence = 'UNCERTAIN'
                
                result['url_analysis'].append({
                    'url': url,
                    'probability': prob,
                    'classification': classification,
                    'confidence': confidence
                })
            
            max_url_prob = max(url_probs)
            
            if max_url_prob >= url_threshold['upper']:
                url_classification, url_confidence = 'PHISHING', 'HIGH'
            elif max_url_prob > url_threshold['lower']:
                url_classification, url_confidence = 'HUMAN_REVIEW', 'UNCERTAIN'
            else:
                url_classification, url_confidence = 'LEGITIMATE', 'HIGH'

        # Final decision logic
        if urls and is_url_only:
            result['final_prediction'] = url_classification
            result['final_probability'] = max_url_prob
            result['confidence_level'] = url_confidence
            result['analysis_method'] = 'URL_ONLY'
        
        elif not urls:
            result['final_prediction'] = text_classification
            result['final_probability'] = text_prob
            result['confidence_level'] = text_confidence
            result['analysis_method'] = 'TEXT_ONLY'
        
        else:
            # Both text and URLs present
            has_trusted_url = False
            has_safe_url = False
            has_phishing_url = False
            
            for url_analysis in result['url_analysis']:
                if url_analysis['classification'] == 'PHISHING':
                    has_phishing_url = True
                elif url_analysis['classification'] == 'LEGITIMATE':
                    if url_analysis['probability'] <= 0.011:
                        has_trusted_url = True
                    else:
                        has_safe_url = True
            
            if has_phishing_url:
                result['final_prediction'] = 'PHISHING'
                result['final_probability'] = max_url_prob
                result['confidence_level'] = 'HIGH'
                result['analysis_method'] = 'HYBRID (Phishing URL Override)'
            
            elif has_trusted_url and text_classification == 'PHISHING':
                result['final_prediction'] = 'LEGITIMATE'
                result['final_probability'] = 0.01
                result['confidence_level'] = 'HIGH'
                result['analysis_method'] = 'HYBRID (Trusted URL + Suspicious Text)'
                result['warning'] = f"Text appears suspicious (prob: {text_prob:.4f}) but URL is from trusted domain"
            
            elif has_safe_url and text_classification == 'PHISHING' and not has_phishing_url:
                result['final_prediction'] = 'LEGITIMATE'
                result['final_probability'] = max_url_prob
                result['confidence_level'] = 'HIGH'
                result['analysis_method'] = 'HYBRID (Safe URL + Suspicious Text)'
                result['warning'] = f"Text appears suspicious (prob: {text_prob:.4f}) but URL appears safe"
            
            else:
                result['final_prediction'] = url_classification
                result['final_probability'] = max_url_prob
                result['confidence_level'] = url_confidence
                result['analysis_method'] = 'HYBRID (URL Primary)'

        # Set recommendation
        if result['final_prediction'] == 'PHISHING':
            result['recommendation'] = 'BLOCK - Phishing detected'
        elif result['final_prediction'] == 'LEGITIMATE':
            result['recommendation'] = 'SAFE - Legitimate content'
        elif result['final_prediction'] == 'HUMAN_REVIEW':
            result['recommendation'] = 'MANUAL REVIEW - Requires human analysis'
        else:
            result['recommendation'] = 'ERROR - Unable to classify'
        
        return result
    
    def predict_batch(self, data):
        if isinstance(data, pd.DataFrame):
            if 'text' in data.columns:
                texts = data['text'].tolist()
            elif 'Email Text' in data.columns:
                texts = data['Email Text'].tolist()
            else:
                text_col = data.select_dtypes(include=['object']).columns[0]
                texts = data[text_col].tolist()
        else:
            texts = data
        
        results = []
        for i, text in enumerate(texts):
            try:
                result = self.predict_single(str(text))
                result['index'] = i
                results.append(result)
            except Exception as e:
                results.append({
                    'index': i,
                    'text': str(text),
                    'error': str(e),
                    'final_prediction': 'ERROR',
                    'final_probability': None
                })
        
        return results



## Example run

The cell below defines the detector, then runs 12 hand-written messages designed to exercise every decision path: trusted URLs, phishing URLs, URL-only inputs, text-only inputs, and both hybrid overrides.


In [2]:
# Test the detector
def test_detector():
    detector = UnifiedPhishingDetector()
    
    test_cases = [
        "Visit our website: https://google.com/search?q=phishing+detection for more information",
        "Check out this GitHub repository: https://github.com/microsoft/ml-for-cybersecurity and this link https://banking-security.tk/update-details.php?account=urgent",
        "Check out this suspicious link: https://secure-paypal-login.com/verify-account?urgent=true",
        "Click here: https://banking-security.tk/update-details.php?account=urgent",
        "https://www.linkedin.com/safety/go?url=https%3A%2F%2Fbit.ly%2F43PtLKD&trk=flagship-messaging-web&messageThreadUrn=urn%3Ali%3AmessagingThread%3A2-OGI5MzdlMGMtYzliYi00MGQ4LThkZGMtOTBiYmZlNzI2NWIzXzEwMA%3D%3D&lipi=urn%3Ali%3Apage%3Ad_flagship3_messaging_conversation_detail%3BM5BP02z%2BQkG0OxNvFUZCAQ%3D%3D",
        "please consider this request. https://banking-security.tk/update-details.php?account=urgent",
       
        "URGENT: Your account will be suspended unless you verify your identity immediately!",
        "Congratulations! You've won $1,000,000! Send us your bank details to claim your prize.",
        "Hi, Thank you for showing interest in hiring me. I completed my Bachelor's in Software Engineering.",
        "Hi Salam, hope your week is going well! I'm reaching out from HR regarding the updated direct deposit form we need all employees to resubmit before the next payroll cycle on Friday. It's a quick process — just reply with your account and routing number and I'll take care of the rest. Thanks",
        "Hi, following up on our conversation from last week. We've finalized the invoice for the consulting work, but our billing details have changed. Please update your records and process the payment to the new account. I've attached the updated banking information below. Let me know once it's submitted so I can confirm with our accounting team.",
        "Hi, my name is David and I came across your profile while looking for someone with your background. I run a small import business and I'm looking for a reliable local partner to help receive and forward payments on my behalf — nothing complicated, and I'd pay a 10%. commission per transaction. Would you be open to a quick call to discuss? No pressure at all, just exploring options."
    ]
    
    print("Results:")
    
    for i, text in enumerate(test_cases, 1):
        print(f"\nTest {i}: {text[:60]}...")
        result = detector.predict_single(text)
        print(f"Prediction: {result['final_prediction']}")
        print(f"Probability: {result['final_probability']:.4f}")
        print(f"Method: {result['analysis_method']}")
        print(f"Confidence: {result.get('confidence_level', 'N/A')}")
        
        if result['contains_urls']:
            print(f"URLs found: {len(result['urls_found'])}")
            for url_analysis in result['url_analysis']:
                print(f"   URL: {url_analysis['url']}")
                print(f"      Probability: {url_analysis['probability']:.4f}")
                print(f"      Classification: {url_analysis['classification']}")
                print(f"      Confidence: {url_analysis.get('confidence', 'N/A')}")

        if result.get('warning'):
            print(f"Warning: {result['warning']}")

if __name__ == "__main__":
    test_detector()

Results:

Test 1: Visit our website: https://google.com/search?q=phishing+dete...
Prediction: LEGITIMATE
Probability: 0.0100
Method: HYBRID (Trusted URL + Suspicious Text)
Confidence: HIGH
URLs found: 1
   URL: https://google.com/search?q=phishing+detection
      Probability: 0.0100
      Classification: LEGITIMATE
      Confidence: HIGH

Test 2: Check out this GitHub repository: https://github.com/microso...
Prediction: PHISHING
Probability: 0.9473
Method: HYBRID (Phishing URL Override)
Confidence: HIGH
URLs found: 2
   URL: https://github.com/microsoft/ml-for-cybersecurity
      Probability: 0.0100
      Classification: LEGITIMATE
      Confidence: HIGH
   URL: https://banking-security.tk/update-details.php?account=urgent
      Probability: 0.9473
      Classification: PHISHING
      Confidence: HIGH

Test 3: Check out this suspicious link: https://secure-paypal-login....
Prediction: PHISHING
Probability: 0.9465
Method: HYBRID (Phishing URL Override)
Confidence: HIGH
URLs found: 1
  